In [2]:
# ── 1. Install & imports ───────────────────────────────────────────────────────
import requests, torch, torch.nn as nn, math, random
device = "cuda" if torch.cuda.is_available() else "cpu"


In [3]:

# ── 2. Data — character-level (65-token vocab vs GPT-2's 50k) ─────────────────
text = requests.get(
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
).text
chars  = sorted(set(text))
vocab_size = len(chars)                         # ~65 — tiny, fast to learn
stoi   = {c: i for i, c in enumerate(chars)}
itos   = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join(itos[i] for i in l)
data   = torch.tensor(encode(text), dtype=torch.long, device=device)
print(f"Vocab size: {vocab_size}  |  Tokens: {len(data)}")


Vocab size: 65  |  Tokens: 1115394


In [4]:

# ── 3. Hyperparameters ────────────────────────────────────────────────────────
block_size = 256
n_embed    = 384
n_head     = 6
n_layer    = 6
dropout    = 0.1
batch_size = 64
max_steps  = 5000
lr_max     = 3e-4
warmup     = 200
eval_every = 200


In [5]:

# ── 4. Model ──────────────────────────────────────────────────────────────────
class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.qkv  = nn.Linear(n_embed, 3 * n_embed, bias=False)
        self.proj = nn.Linear(n_embed, n_embed)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        H, D = n_head, C // n_head
        q, k, v = self.qkv(x).split(C, dim=-1)
        q = q.reshape(B, T, H, D).transpose(1, 2)
        k = k.reshape(B, T, H, D).transpose(1, 2)
        v = v.reshape(B, T, H, D).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(D)
        att = att.masked_fill(self.mask[:T, :T] == 0, float("-inf"))
        att = self.attn_drop(torch.softmax(att, dim=-1))
        return self.resid_drop(self.proj((att @ v).transpose(1, 2).reshape(B, T, C)))

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embed)
        self.att = CausalSelfAttention()
        self.ln2 = nn.LayerNorm(n_embed)
        self.mlp = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed), nn.GELU(),
            nn.Linear(4 * n_embed, n_embed), nn.Dropout(dropout)
        )

    def forward(self, x):
        x = x + self.att(self.ln1(x))
        return x + self.mlp(self.ln2(x))

class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok    = nn.Embedding(vocab_size, n_embed)
        self.pos    = nn.Embedding(block_size, n_embed)
        self.drop   = nn.Dropout(dropout)
        self.blocks = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.ln_f   = nn.LayerNorm(n_embed)
        self.head   = nn.Linear(n_embed, vocab_size, bias=False)
        # Weight tying (saves params + better embeddings)
        self.tok.weight = self.head.weight
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.drop(self.tok(idx) + self.pos(torch.arange(T, device=device)))
        x = self.ln_f(self.blocks(x))
        logits = self.head(x)
        loss = nn.functional.cross_entropy(
            logits.view(-1, vocab_size), targets.view(-1)
        ) if targets is not None else None
        return logits, loss

model = MiniGPT().to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")


Parameters: 10,763,904


In [6]:

# ── 5. Data loader ────────────────────────────────────────────────────────────
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

def get_batch(split="train"):
    src = train_data if split == "train" else val_data
    ix  = torch.randint(len(src) - block_size, (batch_size,))
    x   = torch.stack([src[i: i + block_size]     for i in ix])
    y   = torch.stack([src[i + 1: i + block_size + 1] for i in ix])
    return x, y

@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split in ("train", "val"):
        losses = torch.stack([model(*get_batch(split))[1] for _ in range(50)])
        out[split] = losses.mean().item()
    model.train()
    return out


In [7]:

# ── 6. Optimizer + cosine LR schedule ────────────────────────────────────────
opt = torch.optim.AdamW(model.parameters(), lr=lr_max, weight_decay=0.1, betas=(0.9, 0.95))

def get_lr(step):
    if step < warmup:
        return lr_max * step / warmup
    progress = (step - warmup) / max(max_steps - warmup, 1)
    return lr_max * 0.5 * (1 + math.cos(math.pi * progress))


In [8]:

# ── 7. Training loop ──────────────────────────────────────────────────────────
for step in range(max_steps):
    # LR schedule
    lr = get_lr(step)
    for g in opt.param_groups:
        g["lr"] = lr

    xb, yb = get_batch("train")
    _, loss = model(xb, yb)
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # ← gradient clipping
    opt.step()

    if step % eval_every == 0:
        stats = estimate_loss()
        print(f"step {step:4d} | lr {lr:.2e} | train {stats['train']:.3f} | val {stats['val']:.3f}")



step    0 | lr 0.00e+00 | train 4.282 | val 4.279
step  200 | lr 3.00e-04 | train 2.389 | val 2.415
step  400 | lr 2.99e-04 | train 1.989 | val 2.068
step  600 | lr 2.95e-04 | train 1.705 | val 1.866
step  800 | lr 2.89e-04 | train 1.548 | val 1.731
step 1000 | lr 2.80e-04 | train 1.453 | val 1.658
step 1200 | lr 2.69e-04 | train 1.379 | val 1.605
step 1400 | lr 2.56e-04 | train 1.325 | val 1.559
step 1600 | lr 2.41e-04 | train 1.281 | val 1.538
step 1800 | lr 2.25e-04 | train 1.238 | val 1.515
step 2000 | lr 2.07e-04 | train 1.196 | val 1.512
step 2200 | lr 1.89e-04 | train 1.171 | val 1.493
step 2400 | lr 1.70e-04 | train 1.137 | val 1.498
step 2600 | lr 1.50e-04 | train 1.112 | val 1.487
step 2800 | lr 1.30e-04 | train 1.081 | val 1.493
step 3000 | lr 1.11e-04 | train 1.057 | val 1.500
step 3200 | lr 9.26e-05 | train 1.035 | val 1.502
step 3400 | lr 7.50e-05 | train 1.011 | val 1.510
step 3600 | lr 5.87e-05 | train 0.995 | val 1.503
step 3800 | lr 4.39e-05 | train 0.975 | val 1.505


In [12]:
# ── 8. Generation with top-k sampling ─────────────────────────────────────────
@torch.no_grad()
def generate(prompt="ROMEO:", steps=500, temperature=0.8, top_k=40):
    model.eval()
    x = torch.tensor([encode(prompt)], device=device)
    for _ in range(steps):
        idx_cond = x[:, -block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature
        # Top-k filtering — suppress all but the k most likely tokens
        if top_k:
            v, _ = logits.topk(top_k)
            logits[logits < v[:, [-1]]] = float("-inf")
        probs  = torch.softmax(logits, dim=-1)
        next_t = torch.multinomial(probs, 1)
        x = torch.cat([x, next_t], dim=1)
    return decode(x[0].tolist())

print(generate("who "))

who of the king
Is continue to the crown, whose king
The beauty of this officer of our soul.
But, you are all your innocent lack'd your king,
Else you will be this from the way
Of the heavens of the house of Lancaster,
The worthiest Warwick shall know you have left
A beggar treason of this displeasure.

BLUNT:
It was a colour that made you fortune,
You know not stands of.

GLOUCESTER:
I might you concerning now, some content
To such a duke: your mother cannot show
Your heart will cut your guests.

L


In [13]:
torch.save(model.state_dict(), "minigpt.pth")